## load libraly

In [ ]:
!pip install -U nemo_toolkit['all']

In [ ]:
!pip install librosa soundfile

In [ ]:
!pip install transformers torch accelerate

In [ ]:
!pip install yt-dlp
!apt-get install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
import yt_dlp
import os

def download_and_convert_to_wav(youtube_url, output_name="audio_test"):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': 'temp_audio', # ชื่อไฟล์ชั่วคราว
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])

    final_filename = f"{output_name}.wav"
    os.system(f"ffmpeg -i temp_audio.wav -ar 16000 -ac 1 -y {final_filename}")

    # ลบไฟล์ชั่วคราว
    if os.path.exists("temp_audio.wav"):
        os.remove("temp_audio.wav")

    print(f"✅ แปลงไฟล์สำเร็จ! ไฟล์ของคุณคือ: {final_filename}")
    return final_filename

# --- ใช้งานจริง ---
url = "https://www.youtube.com/watch?v=Sjwkhso9h8U"
audio_file = download_and_convert_to_wav(url, "my_medical_test")

[youtube] Extracting URL: https://www.youtube.com/watch?v=Sjwkhso9h8U
[youtube] Sjwkhso9h8U: Downloading webpage


[youtube] Sjwkhso9h8U: Downloading android vr player API JSON
[info] Sjwkhso9h8U: Downloading 1 format(s): 251
[download] Destination: temp_audio
[download] 100% of   36.10MiB in 00:00:01 at 23.47MiB/s  
[ExtractAudio] Destination: temp_audio.wav
Deleting original file temp_audio (pass -k to keep)
✅ แปลงไฟล์สำเร็จ! ไฟล์ของคุณคือ: my_medical_test.wav


In [ ]:
import nemo.collections.asr as nemo_asr
import torch

# Select processing device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load Typhoon ASR Real-Time model
print("Loading Typhoon ASR Real-Time...")
asr_model = nemo_asr.models.ASRModel.from_pretrained(
    model_name="scb10x/typhoon-asr-realtime",
    map_location=device
)

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cuda
Loading Typhoon ASR Real-Time...
[NeMo I 2026-03-05 08:58:53 mixins:184] Tokenizer SentencePieceTokenizer initialized with 2048 tokens


[NeMo W 2026-03-05 08:58:54 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /data/workspace/warit/nemo-asr/stt_th_conformer_transducer_large/prepare_data/typhoon_cleanser/20250814/Split_gg/train_data_typhoon_asr_realtime.jsonl
    sample_rate: 16000
    batch_size: 8
    shuffle: true
    num_workers: 8
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 30.0
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-03-05 08:58:54 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader

[NeMo I 2026-03-05 08:58:55 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-03-05 08:58:55 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-03-05 08:58:55 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-03-05 08:58:56 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from /root/.cache/huggingface/hub/models--scb10x--typhoon-asr-realtime/snapshots/1aaf2ea81762b311896898f4c4a376d227865076/typhoon-asr-realtime.nemo.


In [ ]:
if audio_file:
    print("🌪️ Running Typhoon ASR Real-Time inference...")


    if hasattr(asr_model, 'change_decoding_strategy'):
      asr_model.change_decoding_strategy(decoding_cfg={'strategy': 'greedy', 'compute_hypothesis_token_set': False})

    # Run transcription
    transcriptions = asr_model.transcribe(audio=[audio_file])

🌪️ Running Typhoon ASR Real-Time inference...
[NeMo I 2026-03-05 08:58:56 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-03-05 08:58:56 rnnt_bpe_models:506] Changed decoding strategy to 
    model_type: rnnt
    strategy: greedy
    compute_hypothesis_token_set: false
    preserve_alignments: null
    tdt_include_token_duration: null
    confidence_cfg:
      preserve_frame_confidence: false
      preserve_token_confidence: false
      preserve_word_confidence: false
      exclude_blank: true
      aggregation: min
      tdt_include_duration: false
      method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
    fused_batch_size: null
    compute_timestamps: null
    compute_langs: false
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    rnnt_timestamp_t

[NeMo W 2026-03-05 08:58:56 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-05 08:58:56 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 0it [00:02, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 67.99 GiB. GPU 0 has a total capacity of 14.56 GiB of which 10.99 GiB is free. Including non-PyTorch memory, this process has 3.57 GiB memory in use. Of the allocated memory 3.42 GiB is allocated by PyTorch, and 19.11 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import time
import soundfile as sf
import torch
import librosa
import os
import gc
import numpy as np

# ตรวจสอบไฟล์
if 'audio_file' in locals() and os.path.exists(audio_file):
    print("🌪️ Running Typhoon ASR Real-Time with Chunking Support...")

    start_time = time.time()

    # ตั้งค่า Decoding Strategy (set to greedy)
    if hasattr(asr_model, 'change_decoding_strategy'):
        asr_model.change_decoding_strategy(decoding_cfg={
            'strategy': 'greedy',
            'compute_hypothesis_token_set': False
        })

    # โหลดไฟล์และคำนวณขนาด
    signal, sr = librosa.load(audio_file, sr=16000)
    audio_duration = len(signal) / sr
    chunk_size_sec = 30  # แบ่งทีละ 30 วินาที เพื่อคุม RAM ให้ใช้ไม่เกิน 2-3GB
    all_text = []

    # ลูปแบ่งไฟล์ประมวลผล (หัวใจสำคัญในการแก้ OOM)
    for start in range(0, int(audio_duration), chunk_size_sec):
        end = min(start + chunk_size_sec, int(audio_duration))
        chunk = signal[int(start * sr) : int(end * sr)]

        # Normalize เสียงให้ดังพอดี
        if np.max(np.abs(chunk)) > 0:
            chunk = chunk / np.max(np.abs(chunk))

        temp_path = f"temp_chunk_{start}.wav"
        sf.write(temp_path, chunk, sr)

        try:
            with torch.no_grad():
                # ส่งแค่ก้อน 30 วินาทีเข้าไป
                res = asr_model.transcribe(audio=[temp_path])

                # แกะข้อความออกมา
                raw_out = res[0]
                while isinstance(raw_out, list) and len(raw_out) > 0:
                    raw_out = raw_out[0]

                text = raw_out.text if hasattr(raw_out, 'text') else str(raw_out)

                if text.strip():
                    all_text.append(text)
                    print(f"✅ ประมวลผลสำเร็จช่วง {start}-{end}s")
        finally:
            if os.path.exists(temp_path):
                os.remove(temp_path)

        # เคลียร์ RAM ทุกรอบที่จบ Chunk
        torch.cuda.empty_cache()
        gc.collect()

    # สรุปผลประสิทธิภาพ
    full_text = " ".join(all_text)
    processing_time = time.time() - start_time
    rtf = processing_time / audio_duration

    print("-" * 30)
    print(f"⚡ Processing time: {processing_time:.2f}s")
    print(f"🎵 Audio duration: {audio_duration:.2f}s")
    print(f"📊 Real-time factor: {rtf:.2f}x")
    print("\n--- ผลการถอดความ ---")
    print(full_text)

else:
    print("❌ ไม่พบไฟล์ audio_file กรุณาตรวจสอบการดาวน์โหลด")

🌪️ Running Typhoon ASR Real-Time with Chunking Support...
[NeMo I 2026-03-05 09:04:57 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-03-05 09:04:57 rnnt_bpe_models:506] Changed decoding strategy to 
    model_type: rnnt
    strategy: greedy
    compute_hypothesis_token_set: false
    preserve_alignments: null
    tdt_include_token_duration: null
    confidence_cfg:
      preserve_frame_confidence: false
      preserve_token_confidence: false
      preserve_word_confidence: false
      exclude_blank: true
      aggregation: min
      tdt_include_duration: false
      method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
    fused_batch_size: null
    compute_timestamps: null
    compute_langs: false
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    rnnt

[NeMo W 2026-03-05 09:04:59 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-05 09:04:59 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  1.93it/s]


✅ ประมวลผลสำเร็จช่วง 0-30s


[NeMo W 2026-03-05 09:05:00 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-05 09:05:00 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  2.13it/s]


✅ ประมวลผลสำเร็จช่วง 30-60s


...

✅ ประมวลผลสำเร็จช่วง 2640-2670s


[NeMo W 2026-03-05 09:06:47 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-05 09:06:47 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  2.16it/s]


✅ ประมวลผลสำเร็จช่วง 2670-2700s


[NeMo W 2026-03-05 09:06:48 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-03-05 09:06:48 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00, 13.96it/s]


------------------------------
⚡ Processing time: 112.04s
🎵 Audio duration: 2701.78s
📊 Real-time factor: 0.04x

--- ผลการถอดความ ---
การออกกําลังกายไม่ได้ทําให้พอร์ตทําไมอ่ะมันคือการให้ทําให้เป็นโรคกับพอร์ตนะการที่คุณเป็นโรคกับพอร์ตอ่ะเป็นจักรทําไมลดน้ําหนักได้แต่สุดท้ายก็กลับมาเยอะพอเป็นวัยทองอ่ะแล้วบอกว่าน้ําหนักขึ้นง่ายจังพอโมท Excostion อ่ะเป็นสิ่งสําคัญถ้าคุณต่ําลงมันจะทําให้แฟลตที่อยู่ตรงกลางร่างกายอ่ะสตอมากขึ้นก็คืออ้วนลงที่นั่นเองสิบปีที่ผ่านมามันมีการเปลี่ยนพฤติกรรมครั้งยิ่งใหญ่ของมนุษย์ก็คือเรากินมากกว่าเดิมเมื่อปีสองพันอ่ะคนไทย เป็นโรคเบาหวานประมาณหนึ่งจุดห้าล้านคน แต่ตอนเนี้ยให้ถ่ายคนเป็นเบาหวานกี่คน ตั้งแต่อายุยี่สิบถึงห้าสิบสามสิบปีผ่านไปใช่ป่ะ โดยเฉลี่ยแล้วอะ น้ําตักคนเราอาจจะมากขึ้นประมาณสิบห้ากิโลขึ้นจากอะไร ร่างกายคนเราการเผาผ่านน่ะจริง จริงแล้วมันดีนะดีจนถึงอายุหกสิบเลย แต่ทําไมถ้าเราอายุแบบสามสิบสี่สิบอย่างเงี้ย มันถึงค่อยพืช น้ําอับลนที่ไม่มีน้ําตาลแย่กว่าน้ําอับลนที่มีน้ําตาลซะอีก มันส่งผลกระทบต่อระบบหัวใจ มันลดการทํางานของไตถึงยี่สิบเปอร์เซ็นต์ ถ้าพูดถึง ผู้หญิงท

การออกกําลังกายไม่ได้ทําให้พอร์ตทําไมอ่ะมันคือการให้ทําให้เป็นโรคกับพอร์ตนะการที่คุณเป็นโรคกับพอร์ตอ่ะเป็นจักรทําไมลดน้ําหนักได้แต่สุดท้ายก็กลับมาเยอะพอเป็นวัยทองอ่ะแล้วบอกว่าน้ําหนักขึ้นง่ายจังพอโมท Excostion อ่ะเป็นสิ่งสําคัญถ้าคุณต่ําลงมันจะทําให้แฟลตที่อยู่ตรงกลางร่างกายอ่ะสตอมากขึ้นก็คืออ้วนลงที่นั่นเองสิบปีที่ผ่านมามันมีการเปลี่ยนพฤติกรรมครั้งยิ่งใหญ่ของมนุษย์ก็คือเรากินมากกว่าเดิมเมื่อปีสองพันอ่ะคนไทย เป็นโรคเบาหวานประมาณหนึ่งจุดห้าล้านคน แต่ตอนเนี้ยให้ถ่ายคนเป็นเบาหวานกี่คน ตั้งแต่อายุยี่สิบถึงห้าสิบสามสิบปีผ่านไปใช่ป่ะ โดยเฉลี่ยแล้วอะ น้ําตักคนเราอาจจะมากขึ้นประมาณสิบห้ากิโลขึ้นจากอะไร ร่างกายคนเราการเผาผ่านน่ะจริง จริงแล้วมันดีนะดีจนถึงอายุหกสิบเลย แต่ทําไมถ้าเราอายุแบบสามสิบสี่สิบอย่างเงี้ย มันถึงค่อยพืช น้ําอับลนที่ไม่มีน้ําตาลแย่กว่าน้ําอับลนที่มีน้ําตาลซะอีก มันส่งผลกระทบต่อระบบหัวใจ มันลดการทํางานของไตถึงยี่สิบเปอร์เซ็นต์ ถ้าพูดถึง ผู้หญิงที่ท้องเนี่ย แล้วไปกินน้ําอัดลมพวกหมี จะเพิ่มความเสี่ยงการกําหนดได้ถึงสิบเก้าเปอร์เซ็นต์ สวัสดีครับ ยินดีต้อนรับเข้าสู่ดร.สท็อก Podcast ที่หมอ และผู้เชี่ยวชาญทางด้านสุขภาพจะมาพูดคุยประเด็นเรื่องสุขภาพต่าง ต่างอยู่กับผม หมอจิมมี่ แพทย์ผู้เชี่ยวชาญทางด้านเวชศาสตร์ป้องกัน และหมอเอมี่ แพทย์ผู้เชี่ยวชาญด้านวิชศาสตร์ป้องกัน อ่า วันนี้หมอเอมี่นะครับ เดี๋ยวเราจะมาพูดคุยเรื่องการลดน้ําหนัก ซึ่งเป็นประเด็นที่คุยกันมันเยอะและหลายคนพยายาม ลดน้ําหนักแต่ไม่สําเร็จทั้งที่เรามีความรู้อัปเดตใหม่ตลอดเวลาเนาะรวมถึงยาลดน้ําหนัก Blocho ว่าประเทศไทยเราเนี่ยพูดถึงเรื่องของน้ําหนักเกินน่ะในอาเซียนเนี่ยเราได้อันดับสองนะอันดับสองเลยเหรอลองจากมาเลเซียนะว่ามาเลเซียอ้วนสุดแล้วแต่มันก็คือเราและคนน้ําหนักเกินนะ แล้วพอเรามาดูประชากรของคนไทยที่มีภาวะน้ําหนักเกินเนี่ยอยู่ที่สามสิบสองเปอร์เซ็นต์ของประชากร ก็หมายถึงว่าคนไทยเจ็ดสิบล้านคน สามสิบสองเปอร์เซ็นต์นี่คืออ้วน ไม่สามสิบสองล้าน สามสิบสองเปอร์เซ็นต์ เนี่ยคือน้ําหนักเกิน อ๋อน้ําหนักเกิน แต่เก้าเปอร์เซ็นต์น่ะเป็นโลกอ้วน เก้าเปอร์เซ็นต์เป็นโรค นอกจากนั้นเนี่ยโรคเบาหวานที่มันมากับโลกอ้วนเนี่ย เมื่อปีสองพันอ่ะ คนไทยเป็นโรคเบาหวานประมาณหนึ่งจุดห้าล้านคน อือ แต่ตอนเนี้ยให้ทายคนเป็นเบาหวานกี่คน ถ้าครั้งที่แล้วหนึ่งจุดห้าล้านกี่เท่าดีอ่ะขึ้นกี่เท่า ประมาณสี่ห้าเท่าได้ป่ะ แปดจุดห้าล้านคน ในปัจจุบัน เยอะมากเลย เยอะมากเนาะ เพราะเจ็ดแปดเท่า เออ เจ็ดแปดเท่า เพราะฉะนั้นเนี่ยโลกอ้วนทําไมเรากลับรถไม่ได้สักที อืมจริง จริงก็เป็นคําถามที่น่าสนใจเค้าจริง จริงแล้วปัจจุบันเราก็แบบมีเทคโนโลยีช่วยเยอะแยะเลยนะ เออแต่ว่าคนไข้ก็ยังมาหาเราเยอะอยู่ดี เออบอกว่ามาหาว่าเออหมอคะช่วยเราน้ําหนักหน่วยขานู่นนี่นั่นอะไรอย่างเงี้ย เออยังเยอะอยู่อืมจริง จริงแล้วหรือยังก็คือก่อนอื่นเลยนะเราต้องมาเข้าใจว่าความหิวคืออะไรก่อน เออก็คือว่าความหิวอ่ะมันเป็นเกี่ยวข้องกับเข้าใจว่าร่างกายคนเรามันค่อนข้างซับซ้อนใช่ป่ะเรามีทั้งสารเคมีเยอะแยะเลยแล้วก็มีโฮมูลด้วยแล้วก็โดยส่วนมากความหิวอ่ะเกี่ยวข้องกับ มวลของเราโดยตรงเนาะเมื่อเราบอกว่าเมื่อทองเราว่างใช่ป่ะแล้วก็แบบไม่มีอาหารเงี้ยแล้วก็เริ่มมีเสียงกับกุ๊กเริ่มหิวแล้วเริ่มหิวแล้วไม่อยากกินอันเนี้ยเกิดขึ้นจากโฮโมนที่ชื่อว่าเกรินเกลินน่ะเป็นโฮโมนที่เขาเรียกว่าอะไรผลิตจากในพวกลําไส้แล้วมันก็เพราะอาหารลําไส้ต่าง ต่างเนี้ยมันก็ส่งสัญญาณไปที่สมองแล้วบอกว่าเฮ้ยเธอไม่มีอาหารแล้วนะเธอไม่มีแคลอรี่แล้วนะต้องการกินเออแล้วก็เลยรู้สึกว่าเฮ้ยหิวต้องกินเออพอเรากิน ไปประมาณนึงอ่ะที่มาจากว่าเริ่มอิ่มอ่ะมันก็จะเริ่มส่งสัญญาณว่าเฮ้ยเฮ้ยแฟลตเซลล์ไข่มันเราอ่ะจะผลิตฮอร์โมนที่ชื่อว่าเล็บตินออกมาเล็บติน่าจะเป็นฮอร์โมนให้เราแบบว่าเฮ้ยอิ่มเถอะพอเถอะพอได้แล้วนะคะเออไม่งั้นเราก็จะกินกินแบบไม่มีวันหยุดใช่ป่ะชั้นฮอร์โมนสองตัวเนี้ยเป็นฮอร์โมนที่ว่าเออหิวนะแล้วก็ฮอร์โมนที่ว่าเอออิ่มได้แล้วนะเออซึ่งในงานวิจัยอ่ะก็พบเจอว่าคนปัจจุบันเนี้ยมีการดื้อต่อเล็บตีนค่อนข้างเยอะเหมือนคล้ายดื้อต่ออิน สุริน ใช่เป็นเบาหวาน ใช่แต่ดื้อเล็บตินแทน ใช่ ก็แบบว่าเล็บตินอาจจะสร้างเรานะ แต่ก็ยังหิวอยู่ ซึ่งอาจจะเจอมากกว่าคนที่เป็นโลกอ้วนจริงจังแบบว่ากินแล้วแต่ยังแบบยังกินอีกอ่ะเพราะว่ายังกินต่อกินต่ออย่างเงี้ยก็เพราะว่าฮอร์โมนเล็บตินอ่ะเขามีปัญหาแล้วนอกจากฮอร์โมนสองตัวเราเนี่ยยังมีฮอร์โมนอีกสองตัวที่สําคัญมากก็คือฮอร์โมน Insuring ที่เราพูดไปก่อนหน้านี้เนาะแล้วก็ฮอร์โมนคอร์ดิโซฮอร์โมน Codisholนี่เป็นฮอร์โมนความเครียดใช่ป่ะแต่ว่ามันก็เป็นน่าสนใจว่าเออทําไมแบบตอนกลางคืนอ่ะ เรานอนน่ะ เรานอนประมาณหกถึงแปดชั่วโมงต่อวันใช่ป่ะ แล้วทําไมเราแบบทําไมระหว่างหกถึงแปดชั่วโมงถึงไม่กิน อ่าใช่ เพราะว่าเราอยู่กลางวันเรายังหกแปดชั่วโมง ก็กินนี่คือหิว หกแปด สองสามชั่วโมงก็กินแล้ว เออ ไม่ไม่ต้องหกแปดอย่างเงี้ยแต่คือทําไมเวลาเรานอนอย่างเงี้ย เออเราถึงแบบไม่รู้สึกหิวใช่ป่ะ เขาก็มาทํางานวิจัยแล้วจะว่าเออเวลานอนเนี่ยเรามีโก๊ดฮอร์โมนที่สูงนะ ที่เราก็เอ๊ะสี่ทุ่มถึงตีสองอยู่ต้องนอน เพราะพอเรานอนเราเนี่ยเราก็จะมีโก๊ดฮอร์โมนหลัง ถูกป่ะ Good ฮ่องกงเราก็จะไม่ค่อยหิว เออ แล้วพอเราตื่นขึ้นมาตอนเช้าเนี่ย ประมาณเจ็ดถึง แปดโมงเนี่ยก็จะเป็นช่วงที่ฮอร์โมนคอดิโซสูงสุดเออซึ่งพอฮอร์โมนคอดิโซสูงอ่ะน้ําตามันยังค่อนข้างสูงเพราะน้ําตาลค่อนข้างสูงเราเลยไม่หิวมันเลยลิงก์กันไงแต่เพราะว่าพอเราแบบว่าตื่นปุ๊บเราก็เริ่มทํา activityป่ะฮอร์โมนคอดิโซก็เริ่มดอกน้ําตาลเราก็เริ่มดอกเราจึงเริ่มรู้สึกหิว เราเริ่มเริ่มกิน ใช่แต่ว่าอีกประเด็นนึงอ่ะที่อยากคุยก็คือว่าความหิวกับความหย่า เออถ้าเป็นภาษาอังกฤษแล้วบอกว่าฮังเกอร์แอนด์แอปพลิเคชิตไทยภาษาไทยก็ต้องบอกว่าความหิวกว่าความอยากเพราะว่าจริง จริงแล้วคนเราปัจจุบันน่ะรู้หรือเปล่า เผลอว่าเราหิวหรือเปล่า เราอาจจะอยากกินเราอยากจะอยากกินแต่ว่าไม่ได้หิวนะ เออเขาก็เลยบอกว่ามีการบอกว่าพูดกันว่า จริงแล้วความหิวอะหมายถึงว่าร่างกายเราต้องการแคลอรี่เสริมเราถึงต้องกินก็คือเป็นฮอร์โมนแบบว่าเกลินสั่งเราถึงต้องกินแต่ปัจจุบันเราอ่ะมีความหยาบมากกว่าความหิวหมายถึงว่าฉันเห็นอันนี้แล้วฉันอยากกินเขาก็เลยบอกว่าเอาง่ายง่ายนะยูเอาสลัดมาจานนึงถ้าสมมุติว่ายูกินสลัดจานนั้นได้แปลว่ายูหิวจริงแต่ถ้าเอาสลัดมาวางแล้วอ่ะแล้วเราไม่กินแปลว่าเราไม่ได้หิวแต่เราไม่อยาก แล้วแค่อยากจะกิน เออ ก็เลยอยากบอกว่า เออมันก็จะมีความอยากกินกว่าความหิว เออที่เราต้องแยกแยะให้ชัดเจน อือ ทําไมลดน้ําหนักได้แต่สุดท้ายก็กลับมา You คืออันนี้มันต้องเข้าใจก่อนว่าร่างกายของเราอ่ะมันมีความสมดุลอยู่แล้วไม่ว่าจะเป็นการกินอาหารเรียกว่า Energy in กับ Enenergy out คือเรื่องของการเผาผน้อเพราะฉะนั้นร่างกายเรามันจะมีคอนเนคชั่นว่าเอ้ยเราควรกินหรือเราไม่ควรกินเหมือน Episodeครั้งที่เราพูดว่ากับ Brand connection เพราะฉะนั้นเนี่ยบางคนน่ะ เวลาหิวก็จะโมโหถูกไหมครับเนาะถ้าเราย้อนกลับมาเนี่ยเราก็จะบอกว่าเอ๊ะเวลาที่เราน้ําหนักตัวเพิ่มมากขึ้นเนี่ยร่างกายมันก็ชินกับการกินอาหารที่เยอะเข้าไปแล้วการเผาผ่านเราก็อยู่ระดับเดินเท่านั้นแต่เมื่อไหร่ที่เราอยากลดน้ําหนักเร็วเกินไปเนี่ยร่างกายมันเกิดอะไรขึ้นมันช็อกไปมันช็อกพอเวลามันเกิดช็อกเนี่ยมันเกิดการเปลี่ยนแปลงของโฮโมนเมื่อกี้หมอเอมี่บอกมาแล้วเรื่องของโฮโมนความหิวโฮโมนการอิ่มพวกเกรนินเล็บติน ใช่พอร่างกายมันลดน้ําหนักเร็วเกินไปไขมันมันลดลงปุ๊บร่างกายเอ๊ะหายไปไหนไขมันงงมันก็เลยรีบหลังโฮโมนเกรนมากยิ่งขึ้นทําให้เราหิวเราก็อยากกินแล้วอยากกินร่างกายอีกอันนึงก็คือไปลดระบบการเผาผ่านกินเยอะไปลดระบบการเผาผ่านและอีกอันนึงคือมีรีวอร์ดคือร่างกายให้รางวัลตัวเองฟู้ดเบเฮเวอร์รีวอร์ดคือทําไมถึงว่าแบบว่าเริ่มจะเครปแล้วอยากกินอยากจะกินอยากกินอาหารมากยิ่งขึ้นแต่ถ้าเราดูจริง จริงอ่ะ ซ้อนกลับไปประมาณหนึ่งพันปีที่แล้ว ร่างกายของเราอ่ะที่จริงมันก็เริ่มกะเก็บอาหารตั้งแต่คือตั้งแต่สมัยนั้นเราเนาะยุคหินป่ะในยุคหินก็คือหนึ่งพันคือสมัยก่อนเนี่ยคือร่างกายเราไม่มีอาหารกินตลอดเวลามีช่วงเวลาที่เราออกล่าคือเราไม่มีคือเราไม่มีตู้เย็นเราไม่มีตู้เย็นเราไม่มีตู้เก็ตเราไม่มีที่เก็บถูกไหมเพราะฉะนั้นเราต้องออกไปล่าอาหารแล้วมีช่วงเวลาที่เราต้องฟาสกับฟาสคือเราต้องล่าแล้วเราต้องอด คือบางเวลาเราก็ไม่มีแบบอาหารที่กินตลอดเวลาร่างกายมันเลยบอกว่าร่างกายบอกร่างกายว่าสมองบอกร่างกายว่าล่านะกักเก็บอาหารบางส่วนเพื่อไว้ใช้ตอนที่เราฟาส อือ ตอนที่ไม่มีของกินไม่มีของกินแต่ปัจจุบันคืออาหารเรามีร้านของชําเราเรียกแก๊ปได้ตลอดเวลาเราสั่งอาหารเราเก็บอาหารได้กินได้ทุกเมื่อเราฟีด Feesfice Facebook ไม่มีฟาสเลยพอเราฟีดไปตลอดเวลาเนี่ยเป็นไงก็อ้วนนะ ก็อ้วน อือ นะครับเนาะ สิบปีที่ผ่านมามันมีการเปลี่ยนพฤติกรรมครั้งยิ่งใหญ่ของมนุษย์ ก็คือเรากินมากกว่าเดิม หมอยังไม่อยู่ตอนนี้ทําไง ตอนนี้ก็สั่งแก๊ปป้าแล้วก็หรือเว็บก็ไปเซเว่นเพราะมันเปิดยี่สิบสี่ชั่วโมง ก็คือกินเลย ใช่ เออ ก็บอกว่ามันเป็นพฤติกรรมที่มันง่ายอย่างเงี้ยแล้วเราก็ติดถูกป่ะอย่างตอนเราอยู่เวรอย่างเงี้ยสมัยก่อนอยู่เวรกลางคืนใช่ป่ะมันก็มีเซเว่นให้กินไงอย่างน้อยถ้าเป็นสมัยก่อนก็คืออาจจะต้องอดเออหรือว่าต้องซื้อขนมปังมาเก็บไว้อะไรอย่างเงี้ยเออแล้วก็จริง จริงแล้วอ่ะมันอีกมีอีกอันนึงนะที่แบบว่า เจอคําถามค่อนข้างบ่อยมากจากคนไข้คือบอกว่าเอ้ยถ้าเราแก่ขึ้นหรืออายุมากขึ้นน่ะน้ําหนักมันต้องมากขึ้นตามป่ะเออก็แบบว่าเป็นอะไรที่น่าสนใจแล้วก็เลยแบบว่าไปค้นคว้างานวิจัยเพิ่มเติมก็เจอว่าถ้าคนเราอ่ะตั้งแต่อายุยี่สิบถึงห้าสิบสามสิบปีผ่านไปใช่ป่ะโดยเฉลี่ยแล้วอ่ะน้ําหนักคนเราอาจจะมากขึ้นประมาณสิบห้ากิโลมันต้องขึ้นอยู่แล้วแหละเออคราวนี้ก็บอกว่าเออสิบห้ากิโลเนี้ยมันขึ้นจากอะไรเพราะว่าจริง จริงก็บอกว่านั่งกันคนเราการเผาผ่านน่ะจริง จริงแล้วมันดีนะดีจนถึงอายุหกสิบเลยแต่ ทําไมถ้าเราอายุแบบสามสิบสี่สิบอย่างเงี้ยมันถึงค่อยเพิ่มอย่างเงี้ยเขาก็เลยบอกว่ามีการทํางานวิจัยเพิ่มเติมเพราะจริง จริงแล้วเป็นเพราะอะไรเนาะอันดับแรกอ่ะที่เราเจอเขาบอกว่าคนเราโดยส่วนมากเมื่ออายุสามสิบห้าปีขึ้นไปอ่ะจะลอสมาร์ทหนึ่งเปอร์เซ็นต์ทุกปี ก็ไม่ถึงว่าถ้าสมมุติเราอายุสามสิบห้าอย่างเงี้ยถึงห้าสิบห้าปีมันก็หายไปสิบห้าเปอร์เซ็นต์ถูกป่ะแล้วก็บอกว่าถ้ามัสเซิลแมสอะหรือว่ากล้ามเนื้อเราหายไปอ่ะกล้ามเนื้อมันเป็นอะไรที่เราอ่ะต้องใช้พลังงานเยอะถูกป่ะโปรเพาผ่านถึงจะดีถ้าคุณมีกล้ามเนื้อพอเราก้ามเนื้อหายไป ก็หมายความว่าคุณกินเท่าเดิมคุณออกกําลังกายเท่าเดิมยังไงคุณเขาอ้วน เพราะว่าระบบขอผ่านคุณน้อยลงนั่นเอง ฉะนั้นการแก้ของเราคือเราต้องออกกําลังกายใช่ป่ะ เอ่อออกกําลังกายฟิตเนตเนาะหรือว่าเราจะออกกําลังกายท่าต่าง ต่างเนี่ยก็ได้โยคะอะไรก็ได้อย่างเงี้ยรู้เวทเทรนด์ก็ได้ทําให้ Musk Mard คุณดีเท่าเดิมกว่าเดิมมากกว่าเดิมอย่างเงี้ยอันนี้คืออันที่หนึ่งเนาะอันที่สองอ่ะมันเป็นไปเจอว่าการวิจัยของญี่ปุ่นอ่ะเมื่อมันปีสองพันยี่สิบเอ็ดเขาก็ทําเซอร์เวย์ทั่วประเทศเนาะแล้วก็เก็บสถิติว่าเฮ้ยอะไรเป็นแฟคเตอร์ทําให้ คนญี่ปุ่นอะ ยี่สิบปีของช่วงอายุอ่ะ มันน้ําหนักเพิ่มขึ้นแล้วเขาก็บอกว่าถ้าสิ่งเหล่าเนี้ย ถ้าคุณทํา จะทําให้น้ําหนักตัวเพิ่มขึ้นขั้นต่ําคือสิบกิโลเลยนะ ในช่วงเวลา ยี่สิบปีก็บอกว่าปีนึงคุณขึ้นประมาณครึ่งโลเนาะ อันดับแรกที่เจอก็คือการอดนอน เพราะว่าบางทีคนเราเนี่ยอาจจะบอกว่าเฮ้ยฉันต้องทํางานหนัก ฉันต้องทํางานเยอะหรือแพทย์เอง อย่างเราหมอเนี่ยก็คือจะอดนอนค่อนข้างเยอะใช่ป่ะ เขาแต่ว่าถ้าคุณนอนน้อยกว่าห้าชั่วโมงนะ อ้วนแน่นอน อันนี้อันที่หนึ่งเนาะ เขาบอกว่าถ้าคุณนอนน้อยอย่างเงี้ย อย่างคอนดิซาเลเวลอย่างเงี้ย ตื่นเช้ามามันก็จะต่ํามันจะไม่สูงถูกป่ะ พอมันต่ําชุก้าก็ต่ําหรือว่าน้ําตาลต่ําเราก็เลยหิว ก็เลยกินบ่อยคราวนี้อันที่หนึ่งเนาะ อันที่สองที่เจอบ่อยมาก มาก เขาบอกว่าการที่คุณไม่กินอาหารเช้า ก็คือทําให้น้ําหนักขึ้นง่ายเหมือนกัน เพราะว่าพอก็พอคุณไม่กินอาหารเช้าอะ เราอาจจะชอบกินจุกจิกไง กลายเป็น riging นะอาจจะไม่ใช่ฮังกรีนะ อาจจะเป็นการเครปวิ่ง กินนู่น กินหนี้จุกจิกเยอะแยะเลย ก็เลยทําให้น้ําหนักเราขึ้นง่าย อันนี้คืออันที่สอง อันที่สามที่เจอมากก็คือพวกทําการออฟฟิศก็คืออะไรนั่งเยอะไง วันนั่งก็ อี้ นั่งทํางานไม่ได้มีเวลาลุกไปเดินเยอะอะไรอย่างเงี้ย ก็เลยบอกว่าถ้านั่งเก้าอี้มากกว่าห้าชั่วโมงต่อวัน หรือเดินได้น้อยกว่าหนึ่งชั่วโมงต่อวัน หนึ่งชั่วโมงเนี่ยถ้าเดินได้ของเราก็ประมาณสักหกเจ็ดพันเก้าต่อวัน ก็ยังก็ยังก็ยังก็ยังก็ยังก็ยังก็ยังก็ยังก็ยังก็ยังก็ยังโอเคใช่ป่ะ แต่ถ้าคุณเดินไม่ได้ขนาดนั้นน่ะ ก็ทําให้คุณน้ําหนักขึ้นค่อนข้างง่าย อันนี้ก็เป็นปัจจัยสองอันใหญ่ที่เจอ แต่ว่าของผู้หญิงอะก็จะเจออีกอันนึงที่เจอได้ง่ายมากอันที่สามก็คือเรื่องข้อมูล คนใครก็ถามว่าเฮ้ยให้มีหมอส่วนใหญ่แบบว่าพอเป็นวัยทองอ่ะ แล้วบอกว่าน้ําหนักขึ้นง่าย เออก็เลยบอกว่าเออถ้าเป็นไหวทองเนี่ยจริง จริงแล้วอะมันก็แค่แบบว่าวูบวาบอะไรอย่างงี้ป่ะแล้วก็แบบว่าเจอตอนหลังก็เจอว่าเออมันนอกจากการอาการวูบวาบนอนไม่หลับแล้วเนี่ยก็ยังมีอาการที่แบบว่าทําให้น้ําหนักขึ้นง่ายด้วยยิ่งคุณบอกว่าจากห้าสิบไปหกสิบเนี่ยจะเป็นช่วงที่น้ําหนักขึ้นค่อนข้างง่ายมากเนาะเขาก็บอกว่าฮอร์โมน Estogen อ่ะเป็นสิ่งสําคัญเขาบอกว่า Esson ถ้าคุณต่ําลงอ่ะมันจะทําให้แฟตที่อยู่ตรงกลางร่างกายอ่ะสตอมากขึ้นก็คืออ้วนลงทุนนั่นเอง ของผู้หญิงจริงผู้ชายจะอาจจะเป็นก่อน แต่ผู้หญิงอาจจะเป็นช่วงง่ายหลังจากที่หมดประจําเดือนเนาะ อันนี้ก็คืออันที่หนึ่งของฮอร์โมน อันที่สองอ่ะที่เป็นค่อนข้างเยอะแล้วผู้หญิงอาจจะไม่ได้แบบไม่ค่อยเข้าใจ ก็คือภาวะที่โลกที่เป็นว่า PCOS หรือโลกถุงน้ําในรังไข่ที่เรามีหลาย หลายก่อน เพราะว่าอะไรผู้หญิงกลุ่มเนี้ยจะมีโฮโมนเพศชายในร่างกายค่อนข้างเยอะ แล้วจริง จริงแล้วผู้หญิงเราควรจะมีโฮมวลเพศหญิงเด่นใช่ป่ะ แต่ว่าผู้หญิงกลุ่มนั้นน่ะอาจจะมีขนยาวสร้างมันเซลล์ง่ายเค้าหน้ามันซิลขึ้นอะไรอย่างเงี้ย เพราะเขามีข้อมูล Test Thos หรือข้อมูลผู้ชายเยอะ ซึ่งข้อมูล Test Today มันก็ไม่ได้เหมาะกับร่างกายผู้หญิงขนาดนั้น เพราะผู้หญิงเหล่านั้นน่ะมันมีข้อมูลค่อนข้างเยอะ มันก็เลยทําให้ Stor Fat ค่อนข้างง่าย ก็เลยทําให้แบบอ้วนง่าย เป็นเบาหวานง่าย อันนี้ก็จะเป็นเหตุผลหลักหลักที่เขาเจอว่าทําไมแบบเราถึงแบบอ้วนได้ง่ายขึ้น อ่าแต่เมื่อกี้หมอเอมี่พูดมาถึงเรื่องของอาหารเช้าถูกไหม คือถ้าเราดูย้อนไปในสมัยของคุณพ่อคุณแม่มันจะมีประโยคนึงที่พูดคําว่า Brexfros Isumpow Sinpowth ren melo of Jay ก็คือหมายถึงว่าอาหารเช้า เป็นมือสําคัญที่สุดที่เราควรจะกินเลยรู้ไหมว่าประโยคนี้มันมาจากไหนมันมาจากซีเรียลอาหารเช้าในยุคเจ็ดศูนย์ อ่าคือโฆษณามันคือการตลาดเพราะฉะนั้นเนี่ยถามว่าคนเราจําเป็นต้องกินอาหารเช้าจริงหรือเปล่า อือ ก็ปกติก็ตื่นมาส่วนเดียวพอตื่นมามันหิวหนึ่งกินไงมันหิวหนึ่งกินถูกไหมทีเนี้ยเขาต้องมาเข้าใจศึกษาของเขาเรียกว่าเซอร์คัเวนด์ Wider ในร่างกายของเราคือนาฬิกาชีวิตทุกคนอาจจะมีนาฬิกาชีวิตหมดเลยนาฬิกาชีวิตของสมอง และของลําไส้เวลาที่เราตื่นมาระบบสมองทํางานฮอร์โมนทํางานการเคลื่อนไหวลําไส้การย่อยการดูดซึมเริ่มทํางานเพราะฉะนั้นช่วงนั้นน่ะพูดง่ายก่อนเที่ยงเป็นช่วงที่เราควรรับประทานช่วงไหนก็ได้ถือว่าดีต่อร่างกายเพราะร่างกายได้การย่อยได้การเผาผ่าแล้วเราก็ได้เอาไปใช้ไม่มีการเก็บอ่ะมันจะค่อนข้างน้อยมากเพราะเราใช้พลังงานทั้งวันแต่ช่วงเวลากลางคืนเนี่ยอย่างเวลาเราที่เรานอนมันจะมีฮอร์โมนอีกตัวนึงเรียกว่า โฮโมน เมลาโทนิน ซึ่งห้าใครไปกินตอนก่อนนอนเนี่ย เมลาโทนจะเป็นโฮโมนที่เธอพักเถอะ เธอนําไส้อย่าดูซึมนะ เพราะฉะนั้นเนี่ยถ้าใครไปกินตอนกลางคืนอ่ะอาหารก็ไม่ย่อยท้องอืดอีกแล้วมันก็กักเก็บไขมันมากยิ่งขึ้น เออแล้วอีกอันนึงที่เจอบอกก็คือถ้าคนไข้กินกลางคืนแล้วนอนอ่ะเจอว่าเป็นแบบว่าอะไรนะกดไหล่ย้อนเยอะด้วย อือ ทีนี้เรามาพูดเรื่องวิธีการลดน้ําหนักที่ยอดฮิตกันบ้างอะไรเวิร์คอะไรไม่เวิร์คแล้วกลไกของแต่ละวิธีมีอะไรบ้าง วิธี Hothitอันแรกอะที่อยากพูดถึงเลยก็คือเป็นการนับแคล่วับแคล้วส่วนใหญ่เป็นอะไรที่แบบทําง่ายใช่ป่ะ แล้วคนเข้าใจคิดว่าเฮ้ยถ้าเราจดต่อวันต่อวันของผู้หญิงอย่างเงี้ย เออแค่นี้ก็โอเคแล้ว แต่ประเด็นคือการนับแคลที่ถูกต้องเป็นยังไง คนอาจจะยังไม่ค่อยเข้าใจ เช่นสมมุติบอกว่าเฮ้ยฉันอยากกินแค่สองร้อยแคลอะ จะกินไงดี ใช่ป่ะก็อาจจะมีผู้หญิงสองคน คนแรกบอกว่าอะไรเฮ่นกินมาฝรั่งทอดก็ได้อะไรอย่างเงี้ยอีกคนนึงบอกว่าเฮ้ยฉันกินแอปเปิ้ลก็ได้ Apple สี่รูปเขากับสองร้อยแคลพอดีโดยเฉลี่ยนะประมาณนี้ คราวนี้ผู้หญิงคนแรก ที่กินมันฝรั่งทอดก็จะได้อะไรได้คาฟาโบไฮเดชใช่ไหมได้โซเดียมเยอะอะไรอย่างเงี้ยแต่ก็คือสองร้อยแคลแต่คือสารอาหารมีแค่นั้นเนาะแล้วก็ได้บวมด้วยอ่าคนที่สองอย่างเงี้ยอ่าที่ที่บีอย่างเงี้ยที่กินแอปเปิ้ลสีลูกก็ได้อะไรได้ทั้งสารอาหารได้ทั้งไฟเบอร์ถูกป้าได้แบบวิตามินนู่นนี่นั่นอย่างเงี้ยฉันสารอาหารทั้งสองคนนี้ไม่เหมือนกันเลยถึงจะได้แคลอรี่เข้าแทนกันก็ตามฉันก็หมายความว่าถ้าบางทีเราเนี่ยเรากินนับแคลอย่างเดียวอ่ะเราอาจจะผอมนะแต่ผอมแบบที่ไม่มีคุณภาพหรืออา อาจจะไม่ลดลงเลยก็ได้เพราะเป็นการกินที่ไม่ถูกวิธีเนาะคราวนี้ก็จะมีคนบางคนอ่ะแบบทําวิธีรัฐบอกว่าจริง จริงแล้วเนี่ยเราอาจจะกินน้ําอัดลมหรือว่าน้ําอะไรโซดาที่ไม่มีน้ําตาลก็ได้ใช่ป่ะหลายคนแกคิดว่าเอ้ยดื่มน้ําอารมณ์แทนก็ได้หรือน้ําบัตรลนที่ไม่มีน้ําตาลแต่จริง จริงแล้วเนี่ยน้ําอับลนที่ไม่มีน้ําตาลเนี่ยแย่กว่าน้ําอับลมที่มีน้ําตาลซะอีกคือน้ําอารมณ์ที่ไม่มีน้ําตาลอ่ะมันมีสารหวานทดแทนที่เรียกว่าสองตัวคือแอสปาแทม และ SK คือสองตัวนี้เนี่ยเพิ่งมาไม่กี่ปีที่ผ่านมาเอามาใช้ในพวกน้ําอัดลมที่ไม่มีน้ําตาลศูนย์แค่ว้าวไม่มีแคลอรี่ไม่มีน้ําตาลก็ดีสิก็บอกว่าฉันน่าจะผอม ฉันน่าจะผอมลดน้ําหนักได้แต่รู้ไหมว่างานวิจัยล่าสุดคบพ้นว่าคนที่กินน้ําอัดลมพวกนี้เนี่ยมันส่งผลกระทบต่อระบบหัวใจมันลดการทํางานของไตถึงยี่สิบเปอร์เซ็นต์ ยี่สิบเปอร์เซ็นต์ ยี่สิบเปอร์เซ็นต์แล้วถ้าพูดถึงผู้หญิงที่ท้องเนี่ยแล้วไป กินน้ําอัดลมพวกนี้จะเพิ่มความเสี่ยงการคลอดก่อนกําหนดได้ถึงสิบเก้าเปอร์เซ็นต์นี่สูงมากเลยนะสูงมากก็คือแปลว่าจริงจริงแล้วถ้าเรากินเนี่ยอาจจะกินได้บ้างถูกป่ะแต่ก็ไม่ควรกินเยอะเนาะแล้วก็อีกอันนึงอ่ะที่เจอก็คือนอกจากวิธีการนับแคลแล้วอ่ะการเลือกชนิดอาหารก็สําคัญเช่นสมมุติว่าแบบอันที่ง่ายนะเช่นการกินไข่แล้วบอกว่าเฮ้ยกินไข่ดีไข่หนึ่งฟงโปรตีนเยอะหกถึงเจ็ดกรรมใช่ป่ะได้วิตามินบีเยอะแยะมากมายอะไรเงี้ย เฮ้ยยิวคนกินไข่แต่คราวเนี้ยเฮ้ยคุณกินไข่ยังไงอ่ะสมมุติเราบอกว่าเฮ้ยคุณกินไข่อย่างถูกวิธีคุณต้องกินไข่ต้มไข่ลวกใช่ป่ะหนึ่งฟองคุณได้แปดสิบแคล้วแต่บอกว่าเฮ้ยอยากกินอะไรที่มันอร่อยกว่านั้นนิดนึงขอเป็นแบบทอดได้ไหมทอดก็คืออันดับแรกเป็นไข่ดาวไข่ดาวเนี่ยจากแปดสิบเขาเพิ่มเป็นร้อยหกสิบแคล้วเลยนะแค่แบบว่าวิธีการทอดเฉยนะแล้วก็ถ้าจากแบบว่าไข่ดาวไปเป็นไข่เจียวอ่ะจากหนึ่งร้อยหกสิบเพิ่มเป็นสองร้อยห้าสิบแคล้วก็แปลว่าอะไรวิธีการกินไข่เหมือนกันเลยอ่ะแต่แค่กรรมวิธีการทําอ่ะมัน ก็ไม่ได้แล้วอันแรกมันก็ต้มแล้วก็ทอดแบบไหนอย่างเงี้ยมันก็จะได้แบบว่าเคลลลี่ไม่เหมือนกันแล้วฉันบางทีการที่เราบอกว่าเฮ้ยคุณกินอาหารชิดนี้ได้แคล้วเท่าเนี้ยมันอาจจะไม่เท่านั้นไงคุณต้องดูกรรมวิธีในการทําในการปรุงด้วยเออว่ามันใช้อะไรยังไงเช่นอาจจะมีคนไข้คนนึงบอกว่าเฮ้ยหมอผมกินอกไก่อกไก่อย่างเงี้ยอกไก่ปั่นป่ะปั่นเฉยใช่ป่ะหรือว่าคุณกินไก่ทอดอย่างเงี้ยมันก็โดนค่อนข้างง่ายใช่ป่ะฉันเราก็เลยต้องดูว่าจริง จริงแล้วคนไข้อ่ะกินอาหาร อะไร แล้วกรรมวิธีการปรุงแบบไหนด้วย อีกวิธีการกินแบบนึงที่เขาเอามาใช้ในการลดน้ําหนักคือคิโตเจนิกได้เอดคีโตเจนิกไดเอ็กซ์ก็คือว่ากินอะไรที่มันมีแต่ไขมันใช่ไหมคะ คือความหมายของคีโตเจนิกหมายถึงว่าเรากินค่าปริมาณของแป้งน้อยมากคือวันนึงกินไม่ถึงประมาณยี่สิบถึงห้าสิบกรัมต่อวันเท่านั้น แล้วที่เหลือเรากินแป้งกับโปรตีน คิโตเจนิกไดเอดเนี่ยแบ่งออกมาเป็นสี่ประเภทด้วยกัน หนึ่งสแตนดคิโตเจนิกไดเอด คือกินเป็นเจ็ดสิบยี่สิบ เจ็ดสิบคือ โปรตีนและสิบเปอร์เซ็นต์ก็คือถามอันที่สองคือทาเทติ คิโตเจนิกได้ Carditive หมายถึงว่าเรากินคาฟแค่เป็นช่วงเช่นก่อนออกกําลังกายที่เหลือเราก็กินเป็นคีโตเจนิกเหมือนเดิม อันที่สามไซส์คลิ่ง ไซคลิ่ง Kogent หมายถึงว่าห้าวันคีโตอีกสองวันปกติ และอันที่สามก็คือไฮโปรตีนคิโตเจนิกได้ก็คืออาจจะปรับเปลี่ยนเปอร์เซ็นต์เต็ดของพวกแฟช แล้วก็ กับโปรตีนอย่างเช่นอาจจะแฟลตเหลือหกสิบเมื่อกี้เจ็ดสิบถูกไหมสเตจาดแล้วก็ไปเพิ่มโปรตีนเอา ซึ่งคีโตเจนิกได้เอดเนี่ยมันเนี่ยไปช่วยทําให้ร่างกายของเรามันอยู่ในคีโตสิสเตจ ร่างกายจะผลิตคีโตนมากขึ้นที่ตับแล้วคีโตนเนี่ยเป็นเหมือนอาหารของสมองรู้ไหมว่าสมองใช้คีโตนกี่เปอร์เซ็นต์ในการเป็นพลังงานของอ่ะของสมองของเราเจ็ดสิบป่ะ คือที่จริงเป็นห้าสิบห้าสิบเลยห้าสิบห้าสิบเลยสมองเราอ่ะใช้คีโตนประมาณห้าสิบเปอร์เซ็นต์ในการห้าพลัง อ่าแล้วก็น้ําตาลอีกห้าสิบเปอร์เซ็นต์ แต่ที่จริงสมัยเนี้ยคือถ้าเราไม่ได้กินคิโตเจนิกอะไรเอ็นคนก็จะกินแป้งเยอะ แล้วแป้งจํานวนแป้งที่เลี้ยงอ่ะสมองเนี่ยมันเยอะเกินไปแทบคีโตแทบจะไม่มีเลย เพราะฉะนั้นเนี่ยคนอาจจะมีอาการสมองเบลอคิดอะไรไม่ค่อยออกแล้วพอเปลี่ยนไปเป็นคีโตเจนิกไดเอดเฮ้ยสมองแล่นมากขึ้นดีมากขึ้นได้มากขึ้นแล้วที่จริงบางโลกอะก็เหมาะสําหรับคีโตเจนิกไดเอดซะด้วยซ้ํา ใช่เพราะว่าถ้าจากเปเปอร์ส่วนใหญ่แล้วอะจริง จริงอาหารคีโตเจนิกไดเอดอะเหมาะกับคนไข้ ที่เป็นพวกลมชัด แต่คนก็เอามาประยุกต์เนาะอยากลดน้ําหนัก อือ กินบ้าง อือ ซึ่งการวิจัยหลายวิจัยอ่ะคือในช่วงระยะสั้นคิโตเจนิกอาจมาใช้ได้ในการลดน้ําดับแต่ไม่ยั่งยืนแล้วเราไม่สามารถกินคิโตเจนิกในระยะยาวค่อนข้างแบบน้ายาวได้ แต่คิโตเจนกเอจจริง จริงเหมาะสําหรับคนที่เป็นเบาหวานประเภทสอง ซึ่งพอคนที่เป็นเบาหวานมาก มากเนี่ย หนึ่งเกิด Instrin resist the the the thent คือดื้อต่อ Instoring และน้ําตาลที่สูงเกินไป พอเปลี่ยนตัวเองมากินคิโตสนิทในอิทธิ หรือใครมันเยอะเนี่ย งานวิจัยคนพบว่าทําให้ Insulling เนี่ยเซนซิติ หรือลดหมายถึงว่าการทํางานของ Instilling ดีขึ้นถึงเจ็ดสิบห้าเปอร์เซ็นต์ แต่หมายถึงว่า Insulling มันเซนต์กับน้ํา มันเซนต์กับน้ําตาลแล้วทําให้การเวลาไปเจาะเลือดเนี่ย ค่าน้ําตาลสะสมในเลือดอะลดด้วย คือค่า คือหมอก็มีคนไข้เหมือนกันที่เป็นเบาหวาน แล้วก็ไปเจาะค่าเนี่ย ค่า HBC ตอนแรก มาได้เก้า สูงมากหมอก็เลยแนะนําว่าเออลองปรับเปลี่ยนตัวเองไหม หนึ่งออกกําลังกาย สองปรับเปลี่ยนเรื่องของอาหาร ซึ่งการปรับเปลี่ยนอาหารของเขาเราก็เลยแนะนําให้กินเป็นคิโตเจนิกไดเอ็ด ระยะเวลาประมาณสองเดือนกว่า พอกลับมาเจอกันอีกทีหนึ่งน้ําหนักลด จากเดิมน้ําหนักคือประมาณแปดสิบห้าลดไปห้ากิโล แล้วอันที่สองคือค่า Ancy เนี่ย จากเดิมประมาณเก้า ลดเหลือหกจุดห้าก็ถือว่าดี ก็ถือว่าดีมากเพราะในระยะเวลาแค่สองเดือนคือยายังกินปกติเหมือนเดิมแค่เราแค่ปรับเปลี่ยนของ เรื่องอาหารและเพิ่มการออกกําลังกายเข้าไป แต่ว่าจริง จริงแล้วการกินคีโตอ่ะมันก็ต้องแบบว่ามีการเลือกอาหารป่ะเพราะบางคนอาจจะบอกว่าเฮ้ยให้ฉันกินคีโตฉันก็กินแบบอะไรนะแบบว่าพวกสามชั้นได้ อ่า ใช่เราก็ต้องเลือกเนาะไขมันมันมีทั้งไขมันดีและไขมันเร็วเราก็ควรเลือกเป็นกินไขมันดีมากกว่าไขมันเร็วคือเฮลตี้คิโตซินิกไดเอดไม่ใช่ว่าโอ้โหคีโตเบคอนเลย Becon อย่างเดียวน้ํามันทดสามชั้นอย่างเดียวสามชั้นหมูกรอบเบี่ยงเดียวอันนี้เป็น Bashit Dige อืมแปลว่าถ้าสมมุติว่าคิโตเจนิก Digit ที่มันแบบว่าค่อนข้างเคลตี้กับเราก็อาจจะเป็นพวกแบบ Aw Call อะไรเงี้ยบ้าง เออก็อาจจะมีแบบว่าพวกธัญพ่อบ้างน้ํามันประกอบบ้างน้ํามันปลาอะไรอย่างงี้เนาะ อือก็จะค่อนข้างโอเคแล้วก็อีกอันนึงอ่ะนอกจากว่าการกินคีโตแล้วอ่ะอีกอันนึงที่แบบอาจจะเป็นฮอตฮิตของผู้หญิงมากกว่าเนาะคือการกินน้ําผลไม้บอกว่าจริง จริงแล้วมีคนไข้อ่ะหลายคนมากนะที่เจอก็คือคนไข้เบาหวานคนไข้ที่เป็นเก๊าอย่างเงี้ยคนไข้ที่เป็นไขมันพอกตับนะ เป็นคนที่ชอบกินน้ําผลไม้มากเพราะอะไรรู้ป่ะเพราะว่าคนไทยชอบคิดว่ากินผลไม้แล้วดีคือผลไม้อ่ะมันดีแต่ประเด็นคุณก็ต้องคิดว่าผลไม้แบบไหนที่มันมีน้ําตาลเยอะน้ําตาลน้อยอะไรอย่างเงี้ยสมมุตินะว่าเราอยากกินน้ําส้มน้ําส้มหนึ่งแก้วคนไทยน้ําส้มเนี่ยก็ไม่ใช่กินน้ําส้มเฉยเฉยใส่อะไรใส่น้ําตาลใส่น้ําเชื่อมใส่เกลือเพิ่มแค่แบบว่าให้มันอร่อยถูกป่ะเขาก็เลยมีงานวิจัยว่าจริงจริงแล้วอ่ะน้ําส้มแก้วนึงอ่ะน้ําตาลเทือกท่อกับแบบน้ําอารมณ์หนึ่งกระป๋องเลยก็คือประมาณแบบยี่สิบกรัม ซึ่งวันหนึ่งอ่ะเราบอกว่าคุณสามารถกินน้ําตาลได้แค่ยี่สิบสี่กรัมเพอร์เดย์ทั้งมันเลยนะคุณกินน้ําส้มแก้วนึงก็คือแบบว่าอันอื่นคุณไม่ต้องกินแล้วอย่างเงี้ยเออมันมันก็ไม่โอเคใช่ไหมฉันก็เลยบอกว่าแทนที่เราจะกินแบบว่ากินน้ําส้มอย่างเงี้ยอาจจะกินเป็นส้มแทนเพราะการกินส้มอ่ะคุณต้องเคี้ยวใช่ป่ะเคี้ยวแล้วก็กว่าจะกลืน กลืนเสร็จแล้วเนี่ยยังมีพวกไฟเบอร์อะไรต่าง ต่างอีกอะไรอย่างเงี้ยมันก็เลยเป็นการเหมือนกับว่าทําให้ได้สารอาหารครบกว่าแล้วก็ได้พลังงานที่น้อยกว่าอันเนี้ยสําคัญมากแต่ว่าจริง จริงแล้วเนี่ยการที่ เราจะเลือกการกินอ่ะมันไม่ใช่แค่แบบว่าเฮ้ยจับเป็นจากแบบเขาเรียกอะไรเป็นจากน้ําซ่อมเป็นแค่ซ่อมเฉยอ่ะเราต้องมีการศึกษาเพิ่มเติมอีกนิดนึงว่าเฮ้ยอาหารหรือว่าผลไม้ประเภทไหนที่มีดัชดีน้ําตาลต่ํานิดนึงดัชนีน้ําตาลคืออะไรหมายถึงว่าพอเรากินเข้าไปแล้วเนี่ยร่างกายจะดูดซึมอ่ะแล้วกลายเป็นน้ําตาลในกระแสเลือดได้เร็วมากน้อยแค่ไหนเช่นถ้าสมมุติว่าจิมมี่มาถึงอ่ะจิมมี่กินอะไรกินกินสับปะรดพอจิมมี่กินสับปะรดแป๊บนึงอ่ะน้ําตาลมันขึ้นปึ๊บเลย มากอย่างเงี้ยแทนกลับกันหมอไอมิบอกว่าขอกินฝรั่งแทนถ้าการกินฝรั่งอย่างเงี้ยก็ทําให้น้ําตาน่ะมันขึ้นฉากกว่าทางที่แคลอรี่อาจจะเท่ากันนะแต่น้ําตามันน้อยกว่าค่อนข้างเยอะอันนี้ก็เลยบอกว่าถ้าสมมุติว่าเราจะเลือกการกินน้ําผลไม้สมมุติต้องกินเป็นน้ําผลไม้จริง จริงอ่ะอันดับแรกคือต้องศึกษาก่อนว่าผลไม้แบบไหนควรเอามาทําเป็นน้ําผลไม้เช่นถ้าแบบว่าสับปะรดก็อาจจะเลี่ยงนิดนึงเริ่มผลไม้ที่มีน้ําตาลใช่อาจจะเริ่มเป็นแบบฝรั่งไหมเป็นแก้วมังกรไหมเป็นเบอร์รี่อะไรต่าง ต่างอะไร อย่างเงี้ยที่มีความเปรี้ยวนิดนึงอย่างเงี้ยคือโอเคอันนี้คืออันที่หนึ่งเนาะอันที่สองก็คือพยายามเลี่ยงอะไรพวกแบบเขาเรียกว่าอะไรนะเป็นน้ําผลไม้ที่สกัดกากอ่ะเพราะว่าถ้าคุณเอาก่อนเหลือแต่น้ําอ่ะคุณก็คือเขาเหลือแต่น้ําตาลกันนะใช่คือเหลือคือแบบว่าดื่มน้ําตาลเพียวชั้นการที่เรากินน้ําผลไม้อ่ะจริง จริงเป็นการปั่นอาจจะโอเคกว่าน้องอันนี้คืออันที่สองอันที่สามที่สําคัญมากอ่ะคือคุณไม่ควรใส่อะไรปงแต่งเพิ่มแล้วเช่นแบบว่าคนไทยชอบกินแบบใส่น้ําผึ้งน้ําเชื่อม อย่างเงี้ยคิดว่ามันโอเค แต่จริง จริงแล้วอ่ะคุณอาจจะไม่แบบว่าลืมนับไงเพราะจริง จริงแล้ววันหนึ่งอ่ะคุณกินน้ําตาลเยอะเกินไปหรือเปล่า ฉะนั้นถ้าเรากินประมาณเนี้ยก็คิดว่าน่าจะโอเค เมื่อกี้เราก็พูดถึงเรื่องการนับแคลไปแล้วเนาะคีโตเจนิกไปแล้วเนี่ยแล้วก็การดื่มนะครับนับผลไม้ คราวนี้อีกอันนึงก็คือไฮโปรตีนไดเอด หลายคนอาจจะคิดว่าเออ กินโปรตีนหนึ่งพันแคล้วกินแฟชหนึ่งพันแคลวหรือการกินแป้งหนึ่งพันแคลวอะมันเท่ากันหมดแล้วอันไหนดีที่สุดคือจริงจริงอ่ะพอเรามาดูแล้วเนี่ยการกินโปรตีนหนึ่งพันแคลวอ่ะ มันจะทําให้อิ่มนานมากกว่าเปรียบเทียบกับแฟตกับคาโบไฮเตรเพราะอะไรเพราะการกินโปรตีนโปรตีนเนี่ยเป็นมลใหญ่แล้วใช้เวลาในการย่อยนานในกว่าจะย่อยในกระเพาะไก่ในลําไส้แล้วมันจะเปลี่ยนตัวเองเนี่ยกลายเป็นกรดอัเมโนแมนซิตลองคิดดูเรากินโปรตีนเข้าไปย่อยในอาหารของเราแล้วร่างกายเราจะดูดซึมโปรตีนตัวเนี้ยเปลี่ยนเป็นอัเมโนสิตเข้าหลอดเลือดกระแสเลือดพอกระแสเลือดส่งไปในอวัยวะต่าง ต่างในร่างกายดู ซึมเข้าไปเยาวะอื่น อือ มันต้องเปลี่ยนตัวเองจากกฎ Amanassit มาเป็น ตีน โปรตีนอีกแล้วตอนขั้นตอนทุกขั้นตอนน่ะมันใช้พลังงานหมดเลย อือ หลายคนอ่ะก็เลยเอาว่าเออการกินโปรตีนน่ะเป็นวิธีหนึ่งที่มันเผาผ่านได้ค่อนข้างดีและเราทําให้เราอิ่มนานมากเดิมด้วยเพราะโมเดลมันใหญ่นั่นเองก็แปลว่าสมมุติเรากินในเทียบเท่ากับพลังงานเท่าเทียมกันของอาหารสามอย่างโปรตีนแฟตอย่างเงี้ยกักคาบใช่ป่ะคือการกินโปรตีนคิดว่าน่าจะอ้วนน้อยสุดอ้วนน้อยสุดเพราะว่า แฟตกับคาบเนี่ยกินปุ๊บย่อยปุ๊บเข้าเลยใช้ได้เลยใช้ได้เลยเปลี่ยนแปลงไปได้เลยแต่ว่าคราวเนี้ยเราต้องมาดูกันว่าเอ๊ะแล้วเราจะเลือกโปรตีนกินยังไงแพนเบสหรือ Animal Base ก็ก็ขึ้นกับว่าคนไข้ของเราอยากกินแบบไหนอันไหนเหมาะกับตัวเขาแล้วก็ที่สําคัญเนี่ยถ้าใครอยากคิดมากินไทยแพนเบสโปรตีนต้องระวังเรื่องของโรคไตให้มาปรึกษาแพทย์ก่อนแล้วกันแล้วค่อยว่าเออควรกินโปรตีนเท่าไหร่และแบบไหนเพราะจริง จริงแล้วมัน ก็สําคัญนะเพราะว่าจริง จริงแล้วแบบว่าในยุคปัจจุบันน่ะเรามีแบบว่าช้อยที่เลือกกินเยอะมากใช่ป่ะไม่ว่าจะเป็น Panbase เอยหรือเป็นเวย์อะไรอย่างเงี้ยก็เลยอยากจะรู้ว่าจริง จริงแล้วสําหรับคนไข้คนนึงอ่ะเขาควรเลิกกินอะไรมากกว่า คือถ้าสําหรับรดน้ําหนักนะ อือ คือจริง จริงรถน้ําหนักเนี่ยแพนเบสจะดีกว่าเพราะมันทําให้เรารู้สึกอิ่มมากกว่าแอนิเมลอิ่มมากกว่าอย่างเดียวหรืออิ่มนานด้วยอิ่มมากกว่าและอิ่มนานด้วยเมื่อเทียบกับ Normal WayProty หรือ Anmamal Base Proty อ่าแต่ว่า ถ้าสมมุติว่าคนไข้คนนั้นเนี่ยมีโลกหรือมีอะไรเงี้ยก็คือมันสามารถกิน Panbate ได้ทุกคนเลยป่ะก็คราวนี้เนี่ยอย่างหมอก็มีคนไข้ที่เป็นโรคไทรอยด์เนาะ Thailey ต่ํา Thailat ก็จะอ้วนง่ายระบบการเผาผ่านไม่ค่อยดีเท่าไหร่สําหรับคนที่เป็นโรคไทรอยด์ที่จริงแนะนําอยากให้กินเป็น Animal Base มากกว่าเพราะว่าหนึ่งคุณต้องกินปริมาณที่ถึงต่อวันอ่ะแน่นอนเรื่องของการออกกําลังกายการที่กินแอนเนอร์เบนส์ด้วยความว่ามันมีกรดอเมโนที่เรียกว่าไทโรซีน กรดตัวนี้เป็นศาลตั้งต้นของโคโมลไทลอยพอเราพอคนไข้ตอนแรก อุ๊ยอยากลดน้ําหนักก็ไปกินเป็นแพนส์เบตเพราะมันเป็นอะไรที่ฮิตฮอตแต่ระบบไทรอยด์เจ๊งก็เลยอ้วนก็เลยอ้วนอ้วนกว่าเดิมหรือไม่ก็ระบบไทรอยด์ไม่ดีพังเพราะคราวนี้ก็เลยเอ้ยลองปรับเปลี่ยนมาเป็น Animock แบบโปรตีนทั่วไปจากสัตว์บ้างด้วยและกินถึงด้วยเป็นไฮโปรตีนไดเอตพอปรับเปลี่ยนปุ๊บระบบการไทรอยด์ก็ทํางานปกติดีขึ้นซะด้วยซ้ําอันนี้คือคนที่อยากน้ําหนักแล้วถ้าคนที่อยากสร้างกล้ามเนื้อข้ามเหนือก็ต้องกินเป็นแบบ Anmow แล้วเพราะไอมอลอ่ะให้จํานวน โปรตีนที่เยอะกว่า ก็คือแปลว่าจริง จริงแล้วการเราเลือกกินโปรตีนอ่ะ ต้องเลือกก่อนว่า หนึ่งเราอยากกินเพราะอะไรใช่ป่ะ แล้วก็สองเราก็ต้องดูว่าร่างกายเราเป็นแบบไหนด้วย ถูกต้อง อีกอันนึงที่อยากคุยกับการลดน้ําหนักอ่ะ ก็คือเราพูดมาหลายแบบแล้วใช่ไหมเมื่อกี้พูดถึงเรื่องการกินโปรตีนเนาะ อันนี้ก็เลยอยากพูดเกี่ยวกับการงดอาหารหรือฟลาสติกบ้าง ก็คือตั้งแต่เราเด็กอ่ะ พ่อแม่ก็บอกว่าเฮ้ยเราต้องกินอาหารให้ครบสามมื้อเหมือนที่หมอจิมมี่พูดเนาะว่าเออกินเช้ากินกลางวันกินเยนอย่างเงี้ยแถมบอกว่าถ้าเราอดอาหารบางทีอ่ะจะให้เป็นลบกระเพาะอีกแต่จริงจริงแล้ว งานวิจัยก็บอกว่าจริง จริงแล้วการอดอาหารไม่ได้ทําให้เป็นโรคกระเพาะนะการที่คุณเป็นโรคกระเพาะอ่ะส่วนใหญ่เป็นจากการติดเชื้อแบตทีเลที่ชื่อแฮชไพโรไรมากกว่าเนาะแปลว่าจริง จริงแล้วการอดอาหารอ่ะไม่ได้เกี่ยวข้องกับเรื่องโลกกระเพาะคราวนี้ก็มาพูดว่าทําไมสมัยก่อนน่ะคุณถึงมีการฟาสหรือการงดอาหารโดยธรรมชาติเพราะว่าสมัยก่อนเราไม่ได้มีแบบตู้เย็นไม่ได้มีเทคโนโลยีในการถนอมอาหารฉันเราก็ต้องไปล่าสัตว์เหมือนที่จิมมี่บอกใช่ไหมพลาดสัตว์มาปุ๊บกินปุ๊บก็ต้องไงอาจจะไม่ได้ไปล่าอีกคันทีฉันแล้วก็ต้องมีการอดอาหารตอนนี้สะสม ร่างกายไว้ใช่ป่ะ แล้วศิลปพอไล่อีกครั้งค่อยกิน อันนี้ก็เลยเป็นการงดอาหารโดยธรรมชาติ คราวนี้เมื่อบอกว่าโลกเรามีวิวัฒนาการเจริญมากขึ้นอย่างเงี้ย มันก็เลยทําให้การฟาสเนี่ยลดลง แต่ว่าถ้าเรามาดูกันจริงจริงแล้วอะการฟาสหรือการงดอาหารน่ะ มันแทบจะอยู่ที่ทุก ทุกประเทศ ทุกวัฒนธรรมเลยนะ เพราะมันอยู่ในศาสนา เช่นของเมืองไทยเราอะเป็นศาสนาพุทธเนาะ ศาสนาพุทธเราเนี่ยก็คือเช่นอะไร ถ้าคุณถืนสีแปด ถูกป่ะ คุณไปวัดอะไรอย่างเงี้ย คุณก็ห้ามกินมื้อเย็น เพราะคุณไม่กินมื้อเย็น คุณก็ฟาสและแล้วก็ฟาสได้นานด้วยว่าคุณได้กินเท่าไหร่แค่แบบถึงเที่ยงถูกป่ะแล้วก็จากเที่ยงวันของอีกวันนั้นน่ะไปถึงอีกวันนึงตอนเช้ามันก็หลายชั่วโมงแล้วก็ถือว่าเป็นการฟาสโดยปริยายเนาะแล้วก็ศาสนาอื่น อื่นเงี้ยศาสนาครีศศาสนาอิสลามก็มีการฟาสเหมือนกันฉันก็เห็นว่าการฟาสจริง จริงแล้วมันน่าจะมีประโยชน์กับร่างกายเนาะคราวนี้ก็เลยบอกว่าไปศึกษาเพิ่มเติมเขาก็บอกว่าจริง จริงแล้วร่างกายเราอ่ะการใช้พลังการที่เราพูดตั้งแต่ต้นจะมีสองวิธีง่ายวิธีแรกก็คือกินเสร็จปุ๊บเนี่ยก็ใช้อะไรน้ําตา ใช่ป่ะกินเสร็จปุ๊บใช้น้ําตาลได้ทันทีน้ําตาลก็ใช้เป็นพลังงานแล้วแต่เลย คราวนี้อีกอันนึงก็คือถ้าเมื่อไหร่ล่ะเราหยุดกินแปดชั่วโมงนะ ไม่ใช่กินแบบหยุดสองชั่วโมงจะเบิร์นแฟตไม่นะ คุณต้องหยุดประมาณแปดชั่วโมงขึ้นไปหลังจากนั้นร่างกายเราจะต้องบอกว่าเฮ้ย เคยน้ําตาลตกแล้วไม่มีน้ําตาลให้ตับออกมาเปลี่ยนได้เราเนี้ย แล้วก็เริ่มจะใช้แฟชในการเป็นพลังงาน พอใช้แฟลตเนี่ยแฟตก็จะเปลี่ยนเป็นคีย์โตน ใช่ป่ะ เพราะคีย์โตเนี่ยจะไปให้ที่บอกว่าขึ้นไปสู่สมองทําให้บอกว่าสมองทํางานดีขึ้นโปร่งใสจําอะไร ได้ง่ายขึ้นเนี่ยเขาก็เป็นอาหารที่สมองต้องการฉันจริง จริงแล้วเนี่ยเราก็เลยมีการใช้พลังงานสองรูปแบบเนี้ยมานานมากแล้ว คราวนี้ก็บอกว่าเฮ้ยแล้วถ้าแปดชั่วโมงอะเราเพิ่งจะเริ่มได้คีย์โตเด็กนะเราอ่ะจริงจริงแล้วก็คือว่าถ้าเราฟาสหรืองอาหารเนี่ยก็คือควรจะงดให้ได้เกินแปดชั่วโมง เพราะพอมีงานวิจัยบอกว่าเรางดมากกว่าแปดพอได้ถึงสิบสองเนี่ยคีโตมันจะเริ่มเยอะขึ้นถึงจุดที่ว่าเฮ้ยแบบดีมากเลยสมองทํางานแจ่มใสนู่นนี่นั่นเนาะก็เลยอันเนี้ยโอเคสิบสองชั่วโมงคราวนี้ ก็มีคนถามว่าเฮ้ยแล้วคนเราควรไปมากกว่าสิบสองไหมเนาะ จริงจริงแล้วก็บอกว่าควรไปมากกว่าสิบสองนะ เพราะว่าพอเรามากกว่าสิบสองปุ๊บเนี่ย พอเริ่มที่สิบสามถึงสิบสี่ชั่วโมงปรากฏว่าอะไร เขาเจอว่าคนไข้อ่ะ ที่มีปัญหาเรื่องการอักเสบพวกเจ็บข้ออย่างเงี้ย เป็น Otone อะไรอย่างเงี้ย ปรากฏว่าการอักเสบลดลงชัดเจน เพราะว่าอะไรเพราะว่าเราไม่ได้กิน พอเราไม่ได้กินปุ๊บเนี่ย ฟีรานิคเคอร์เนี่ย หรือว่าสารอนุมูลอิสระมันลดลง ก็เลยทําให้การอักเสบลดลง อันนี้สิบสามสิบสี่ ชั่วโมงเนาะ ฉันเราบอกเฮ้ยแล้วถ้าไปอีกนิดล่ะ ไปอีกนิดเราจะได้อะไร พอเราไปประมาณที่ชั่วโมงที่สิบห้า เขาก็เจอว่าผู้ชายส่วนใหญ่อ่ะ ถ้าคุณฟาสเนาะ หรืองดอาหารถึงสิบห้าชั่วโมงอะ Test Tosorron สูงขึ้นถึงสิบสามเปอร์เซ็นต์ อันนี้สิบสามเปอร์เซ็นต์โดยธรรมชาติที่คุณไม่ต้องให้ยา ไม่ต้องให้อะไรเนาะ ฉันแปลว่าผู้ชายนะคะ ก็คือถ้าเราอยากจะแบบมี Test Today ที่ดีขึ้นน่ะ คนฟาสให้ได้ถึงสิบห้าถึงสิบหกชั่วโมงเนาะ ว่าคุณก็จะได้เหมือนกับว่า Test Tost ดีขึ้น นอกจากเทสโทสดีขึ้นแล้วอะ เรายังได้โกรธฮอร์โมนด้วย Growth Home ก็หมายถึงว่าเราสามารถอะไรสเตยังอายุน้อยหรือไม่ใช่อายุน้อยลงคืออะไรอ่อนไวลงถูกป่ะแล้วก็ร่างกายสามารถซ่อมแซมตัวเองได้ดีขึ้นอันนี้ก็เป็นเหตุผลว่าทําไมเราฟาสแล้วอ่ะเราถึงเด็กลงหรือว่าดูอ่อนกว่าวัยพูดได้ง่ายแบบนั้นเนาะ คราวนี้นอกจากการอ่อนกว่าวัยการที่ได้ Growth Home แล้วเนี่ยก็มีอีก มีนักวิจัยญี่ปุ่นอ่ะเขาเพิ่งได้ Nobbel price เรื่องนี้เองก็คือว่าเป็นพวกโอโตฟักกี้ถ้าเราเคยได้ยิน แต่การที่ได้ Otfake อ่ะคุณต้องฟ้าให้ถึงสิบหกสิบเจ็ดชั่วโมง ขึ้นไปนะ พอคุณฟาสได้นานขนาดนั้นน่ะร่างกายเริ่มไงบอกว่าเริ่มขาด ขาดขาดสารอาหารมาสักพักนึงใช่ไหม ฉันต้องอะไร ฉันต้องขจัดของเสียออกจากร่างกาย คราวเนี้ยจะเป็นเวลาที่ร่างกายเอาเริ่มบอกว่าเฮ้ยตรงไหนวะไม่มีเซลล์หวยหวยมีเซลล์ชะลาอย่างเงี้ยเราก็เริ่มจะขจัดทิ้งละพอทิ้งไปปรากฏว่าพอเราทําอย่างเงี้ยบ่อยเซลล์ที่มันชราก็จะลดลงเซลล์ชะลาเนี่ยเป็นอะไรที่แบบเปลืองพลังงานมากเราให้พลังงานมันไปใช้ป้ากินเข้าไปมันก็ใช้ประโยชน์ไม่ได้เพราะว่าพวกเนี้ยจะให้เกิดโรค มากกว่าที่จะเป็นผลดีด้วยซ้ํามีงานวิจัยว่าเซลล์ชราเหล่าเนี้ยหรือเรียกว่าเซนเนสเลวเนาะทําให้เกิดพวกอะไรพวกมะเร็งได้ง่ายขึ้นเป็นโรคเรื้อรังเช่นโลกเบาหวานโรคอะไรเงี้ยได้ง่ายมากด้วยก็แปลว่าถ้าเรามีการงดอาหารหรือว่าแม้กระทั่งคนที่เป็นแบบว่าเบาหวานเนาะมีการงดอาหารก็ทําให้เบาหวานดีขึ้นหรือเราเรียกว่าอย่างอันอ๋อก็คืออดอาหารแล้วเราทําให้เกิด Othagy แล้ว Othy ก็คือไปเอาเซลล์ที่มันไม่ดีอยู่ในร่างกายเนี่ยเอาออกไปใช่พอคราวนี้พูดถึงเรื่องของฟาสติ้งเนี่ยหลายคนอาจจะได้ยินคําว่า IF Internation Fasting ซึ่ง Infesting เนี่ยแบ่งออกมาเป็นสามประเภทด้วยกัน อันแรกเรียกว่า ushnet Day fasting ก็คืออดอาหารกินแต่น้ํานะฟาสตติ้งคือการที่มีศูนย์แคลอรี่ถูกไหมกาแฟดําได้น้ําได้ทั้งวันไม่กินอะไรเลย วันต่อไปกิน Althent วันเว้นวัน ก็คือกินวันนึงไม่กินวันนึง กินวันนึงไม่กินวันนึง อันที่สองเรียกว่า outhned Modify fasting ate Modefine หมายถึงว่าวันที่เราฟาสินก็เหมือนเดิมไม่กินอะไรเลย และอีกวันนึงอะกินแคลอรี่ ที่ต่ํามาก น้อยมาก น้อยกว่าพันแคล้วอีกต่อวัน และอันสุดท้ายเรียกว่า Perdic Fasting peric Fasting ก็อย่างเช่นทุกวันอาทิตย์ฉันจะฟาสยี่สิบสี่ชั่วโมง หรือว่าประจํา ทุกวันที่เท่าไหร่ของเดือนของทุกอาทิตย์นี้ฉันสฟาสติ้งนะ แต่ถามว่าฟาสติ้งไหนดีที่สุด อือ เอ่อหมอเอมี่ลองไทย บอกยากมาก คือจริง จริงมันไม่มีคําตอบนะว่า Internation Fasting หรือการอดอาหารน่ะแบบไหนดีที่สุดมันขึ้นกับว่าร่างกายสุขภาพคุณตอนนั้นเป็นอย่างไร แล้วคุณ ต้องการจะเอาฟาสซิ่งไปทําเพื่ออะไรแต่ว่าจริง จริงแล้วการอดอย่างเงี้ยมันก็ต้องดูที่สุขภาพคนไข้ด้วยป่ะเพราะว่าไม่ใช่ทุกคนเหมาะสําหรับการฟาสเนาะก็จริง จริงแล้วเนี่ยอีกอันนึงอ่ะนอกจากคนไข้จะถามแล้วว่าเอ้ยหมอแบบเราฟาสได้ไหมอันนึงที่คนไข้กังวลก็คือเขาบอกว่าเฮ้ยถ้าเราฟาสตินอย่างเงี้ยแบบว่ามัสเซอร์แมสฉันไม่หายไปเหรอกล้ามเนื้อฉันไม่หายเหรอเขาก็ฉันกินน้อยอย่างเงี้ยก็แบบมีคนไข้ถามมาค่อนข้างเยอะมากเพราะว่าถ้าคุณฟาสแล้วก็คุณเหมือนกับว่ามัสเซอร์แมสหายไป ก็แปลว่าจริง จริงแล้วอะเราแบบว่า เราก็หมายถึงว่าการ Matboni ซึ่งของเราก็จะต่ําลงเนาะ คราวนี้ก็มีงานวิจัยออกมาค่อนข้างเยอะมากเกี่ยวกับเรื่องนี้จริง จริงเขาก็บอกว่าจริง จริงถ้าคุณน่ะฟาสโดยที่คุณไม่เลือกคุณภาพอาหารน่ะ โดยปกติแล้วจะทําให้ Musser Mand ลดลงจริง เพราะว่าอะไรเพราะว่าหนึ่งสารอาหารลดลงเนาะ เอ่อคือแคลลดลงเนาะแล้วก็สองคือโปรตีนไม่ถึงเพราะโปรตีนไม่ถึงแน่นอนว่ามันเกิดการเบรกดาวน์ของมัสเซลถูกป่ะสลายของกล้ามเหนือไปฉันมัสเซอร์แมสคุณก็จะลดลงแต่ก็มีงานวิจัยบอกว่าแต่ว่าถ้าคุณ มีการเลือกกินแล้วมีคุณมีการเมนเทนเนาะการออกกําลังกายที่เหมาะสมเช่นมีการโปรตีนให้ถึงเนาะคัพดีอะไรดีอย่างเงี้ยบวกกับการที่เราอ่ะแบบออกกําลังกายอย่างเงี้ยให้เหมาะสมกับร่างกายเราอย่างเงี้ยคุณก็จะเมนเทน Mussel ได้ดีโดยที่คุณลูสแฟตหรือว่าทําให้แบบว่าแฟตรลดลงคนไข้ก็จะให้แบบลดน้ําหนักได้ด้วยแล้วก็ยังมี Mussomat ที่ดีได้ด้วยฉะนั้นเราว่ามันขึ้นอยู่กับว่าวิธีการว่าแบบว่าเออคนไข้หนึ่งคือเลือกวิธีการไหนแล้วก็สองต้องดูว่าคนไข้คนนั้นเหมาะกับการลดน้ําหนัก แบบไหนมากกว่า คราวนี้เรามาพูดถึงเรื่องการออกกําลังกายบ้าง การออกกําลังกายไม่ได้ทําให้ผอมนะ คือการออกกําลังกายอย่างเดียวแล้วไม่คุมอาหารน่ะ มันไม่ผอมเลยนะ คือการออกกําลังกายมันดีต่อร่างกายของเรานะทุกคนคงรู้หมดแล้วแหละ โอ้โห มันทําให้เสริมสร้างกล้ามเนื้อ ทําให้เราดูอ่อนไวป้องกัน อูปร่างดี ป้องกันการเป็นโรคเรื้อรังในอนาคต แต่ว่าถ้าคุณไม่ได้คุมอาหารไปด้วยเนี่ย คุณก็ไม่ได้ผอม คือแบบว่าถ้าเราแบบว่ากินตามใจฉันเราออกกําลังกายมันก็ไม่ผอม ไม่ผอมอยู่ดี สมมุติถ้าเราอย่างเช่นเรากินข้าวมื้อหนึ่งหนึ่งพันแคลอรี่ เราจะต้องออกกําลังกายหนึ่งพันแคลอรี่ หมอเอมี่คิดว่าออกกําลังกายหนึ่งพันแคลรี่ทําอะไรบ้าง โอ้โหน่าจะนานมาก เพราะปกติแล้วอะเป็นคนที่แบบว่าเดินเร็วอยู่แล้วอะไรอย่างงี้ใช่ป่ะ ขนาดเดินหนึ่งชั่วโมงอะยังออกได้ประมาณแค่แบบร้อยกว่าสองร้อยอย่างเงี้ย อย่างอย่างยกตัวอย่างหมอน้ําหนักเจ็ดสิบกิโลเนาะ ถ้าต้องการเบอร์หนึ่งพันแคลอรี่อะ ต้องวิ่งประมาณสองชั่วโมง โอ้ยนานมาก สองชั่วโมง สองชั่วโมงนี่คือแบบว่าต้องวิ่งเร็ววิ่งสม่ําเสมอเพจเท่าเดิม แล้วถ้าสมมุติอยากปัดจักรยานก็มากกว่านั้นประมาณสามถึงสี่ชั่วโมงเลยถึงจะปืนได้หนึ่งพันแคลอรี่แต่การกินหนึ่งพันแคลี่ง่ายไหมง่ายมากก็แปลว่าถ้าสมมุติว่าจริง จริงแล้วถ้าเราอยากจะลดน้ําหนักอ่ะก็คือควรจะเริ่มจากอาหารก่อนถูกป่ะแต่ที่เราคุยมากันตั้งแต่ต้นอ่ะไม่ว่าจะเป็นการนับแคลนอย่างเงี้ยกินน้ําผลไม้หรืออะไรอย่างงี้เนาะหรือแม้กระทั่งการฟาสเนี่ยคิดว่าต้องมาคอมบายกันมากกว่าเพราะว่าจริง จริงแล้วคนแต่ละคนก็คือไม่เหมือนกันบางคนเนี่ยอาจจะเป็นประเภทที่แบบว่ากิน คีโตโอเคบางคนอาจจะบอกว่าฉันต้องฟาสดีกว่าหรืออาจจะต้องคอมบายกันเนาะแล้วก็ต้องมีการออกกําลังกายด้วยอย่างที่เราคุยกันไปเพราะว่าอะไรเพราะว่าแต่ละคนไม่เหมือนกันและที่สําคัญน่ะถ้าเราสามารถมีโอกาสอ่ะคิดว่าปรึกษาแพทย์ด้วยก็ดีเพราะว่าบางคนอาจจะมีโรคบางโลกที่เราไม่รู้เช่นอย่างที่เราพูดเรื่องของ ไฮโปนไฮลอยไปเนาะแล้วก็อาจจะมีโลกอื่น อื่นหรือว่าอีกอันนึงที่เจอได้บ่อยนะก็คือแบบเป็นโรคแพ้ภูมิแพ้ที่เยอะมากเพราะว่าถ้าคนไข้บางคนอาจจะแพ้อาหารที่เขาโดยไม่รู้ตัวอะไรเงี้ยก็เลยอาจจะทําให้งอ ง่าย เขาว่าลําไส้มีปัญหา ถ้าเป็นคําแนะนําแล้วกันง่ายสําหรับคนที่สุขภาพแข็งแรง อย่างน้อยหมอก็อยากแนะนําว่าเราคนออกกําลังกายอย่างน้อยหนึ่งร้อยห้าสิบนาทีต่อสัปดาห์เป็นแบบ Moderate Exize นะ Modersiness หมายความว่า Hardled คุณต้องขึ้นเนาะอยู่ที่ประมาณแบบหนึ่งร้อยห้าร้อยยี่สิบถึงหนึ่งร้อยห้าสิบเป็นแบบ Morirate แล้วอันนั้นน่ะถือว่าโอเคสําหรับร่างกายเพื่อในการลดน้ําหนัก อือ ก็ก็ดีแล้วก็อีกอันนึงก็บอกว่าถ้าคนไข้บางคนน่ะถ้าสมมุติว่าแค่อยากลดน้ําหนัก เขาก็บอกว่าจริง จริงแล้วแค่เดินอ่ะก็ได้แล้วนะกี่เก้าอะ เขาบอกว่าเดินเนี่ยขั้นต่ําคือต้องเกินหมื่นเก้าต่อ หมื่นเก้า เกินเก้าต่อวัน อือ เขาบอกว่าแต่ข้าวของคุณบอกว่า ถ้าหมื่นเก้าต่อวันเนี่ยแทบจะเหมือนกับเป็นการแบบว่าเมนเทนนะ แต่ถ้าคุณอยากที่จะลดมากกว่านั้นน่ะคุณต้องเกินให้ได้หมื่นเจ็ดถึงหมื่นแปดพันเก้า เออหมอมีเคก็คือว่าอยากให้คนไข้เนี่ยลดน้ําหนัก ก็เลยบอกว่าเออ ใส่นาฬิกาที่มันจับก้าวได้ แล้วก็อ่ะคุณไปเดินทั้งวัน คราวนี้คนไข้เนี่ยเฮ้ยผลออกมา เกินหมื่นเก้าทุกวันเลยแต่น้ําหนักไม่ลด ก็เลยไปถามคนไข้ว่าทําอะไร เออไปทําอะไรอาหารก็ควบคุมแต่เดินก็ครบหมื่นเก้าแล้วทําไมน้ําหนักไม่ลดคนไข้ก็ยอมสารภาพก็คือทั้งวันเนี่ยไม่ถึงหมื่นเก้าได้แค่ประมาณเจ็ดพันแล้วเหลืออีกสามพันพอกลับบ้านปุ๊บเอานาฬิกาไปให้ลูกใส่ไปให้ลูกคืนก็แบบว่ามันก็เลยบอกไม่ไม่ได้ว่าคนไข้บอกว่าทําจริงไม่ทําจริงว่าเอออันนี้ก็เป็นการโกงเล็กน้อยเรามาพูดถึงยาเราน้ํา บ้าง ปัจจุบันมียาลดน้ําหนักมากมายที่ส่งผลกระทบต่อร่างกายมากหรือไม่ก็น้อย หลายคนอาจจะเคยได้ยินถึงปากการลดน้ําหนัก โอ้ยดังมาก ดังมากแล้วคิดว่ามันทํางานต่อร่างกายเราหรือมันได้ผลได้จริงหรือเปล่า หมอต้องบอกอ่อนว่ายาแต่ละตัวเนี่ยควรได้รับการรับรองจากแพทย์หรือปรึกษาแพทย์ก่อนเนาะ ซึ่งยาสมัยใหม่ถ้าถ้าเรามาดูเนี่ยมียาประมวลทั้งหมดประมาณสามตัวด้วยกันตัวแรกเรียกว่า GLP Restopter ก็คือยาปากการลดน้ําหนักนั่นเองหน้าที่ของมันคือมันหลังโคโมรจาก เอ่อกระเพาะของเราเนี่ยส่งไปที่สมองทําให้เรารู้สึกอิ่มก็คือไม่กินน้ําหนักก็ลดอย่างที่สองเนี่ยเรียกว่าซินนิเคิลซานิเคิลเนี่ยเป็นยาเม็ดเนาะเป็นยาที่มีมานานมากและผลข้างเคียงก็ค่อนข้างเยอะขึ้นไส้อาเจียนแต่ยาตัวนี้คือไปบล็อกไม่ให้ร่างกายเราดูซึมไขมันเราก็จะอุจจาระเป็นน้ํามันเลยเป็นไขมันนิดนึงแล้วตัวสุดท้ายเรียกว่าภูโพลพยอน ภูพยรก็เป็นยาเม็ดเหมือนกันซึ่งยาตัวนี้เนี่ยมันจะทําให้เราไม่หิว อืมก็คือกดความหิวเลยกดความหิวในร่างกายพอกดความหิวปุ๊บเราก็ไม่กิน เออ แต่โดยส่วนมาก ณ ปัจจุบันน่ะถ้าเห็นคนไข้ที่ใช้บ่อยเนาะก็น่าจะเป็นปากการและน้ําหนักมาก ใช่ แต่เราต้องเข้าใจนะปากการลดน้ําหนักอ่ะเหมาะสําหรับบางคนไม่ใช่ทุกคนในการลดน้ําหนักสําหรับคนที่เป็นโรคเบาหวานคนที่มีค่า BMI เกินจริง จริงเป็น OBS จริง อ่าอันนี้ก็คือเหมาะคนไข้เนาะ แล้วก็อีกอันนึงอ่ะนอกจากว่าที่จะใช้ยาวน้ําหนักอ่ะก็คือมีคนไข้หลายคนมาปรึกษากับว่าอาหารเสริมเหมือนกันหรือ Subpriment เนาะว่าอะไรบ้างที่จะช่วยลดน้ําหนักได้อันแรก เลยที่อยากคือเกี่ยวกับเรื่อง Probalic หรือว่าจุลินซีที่ดีอ่ะ เพราะว่ามันมีงานวิจัยหลายอันบอกมากเลยว่า ถ้าคุณมีจุลินทรีย์ที่หวยหรือแย่อาจารย์ให้ไขมันคุณขึ้นได้ง่ายมากนะหรือน้ําหนักขึ้นง่ายก็เลยบอกว่าถ้าคุณมีจุลินทรีย์ที่ดีหรือปรับเปลี่ยน Probalic เนี่ย ก็น่าจะทําให้เราน้ําหนักได้ดีขึ้นอันนี้อันแรกเลย อันที่สองอะถ้าเคยได้ยินก็คือ CLA CLA นี่ก็เป็นอะไรที่ฮิตฮอตเนาะที่ทําให้เหมือนกับว่าร่างกายเบอร์แฟตได้มากขึ้นเพราะว่าเป็นแบบว่ากด เอถึงว่าเป็นไขมันที่สําคัญเนาะที่ต้องใช้ CLA เนี่ยกินได้แต่ว่าบางคนอาจจะกินแล้วแบบว่าขึ้น อาเจียนบ้างอะไรบ้างเนาะ อันที่สามเนี่ย ที่บอกว่าคนที่ส่วนใหญ่ที่ถามก็คือชาเขียว ชาเขียวก็คือแบบมีทั้งกินเป็นชากินที่เป็นสารจักรชาเขียวเรียกว่า EGCG เนาะ EGCG เนี่ยก็คือกินแล้วเนี่ยค่อนข้างโอเคบอกว่าทําให้เบอร์แฟตได้ดีอะไรได้ดีแต่ว่าก็มีข้อระวังว่าถ้าคนที่เป็นโรคตับอ่ะควรจะระวังแล้วทําให้ค่าของตับขึ้นได้ค่อนข้างง่ายสุดท้ายเนี่ยที่ตัวง่าย ง่ายที่อยากให้เสริมก็คือวิตามินบีคอมเพล็กซ์หรือว่าวิตามินบีรวมเพราะว่าคน ส่วนใหญ่อาจจะขาดวิตามินบีค่อนข้างเยอะเพราะอาจจะกินอาหารไม่ครบหรือไม่พอการกินวิตามินบีคอมเพ็กอาจจะช่วยทําให้ร่างกาย berinegy เนาะ ไมโตคอนเดียทํางานได้ดีขึ้นพูดได้ง่ายเลย ก็เลยทําให้เหมือนกับว่าทําให้เผาผลาญแล้วก็ลดน้ําหนักได้แต่ถ้าคุณกินวิตามินบีเยอะไปจะทําให้การการเจริญอาหารเนอะอยากกินอาหารมากขึ้นเราอวนขึ้นง่ายอันนั้นก็จะเป็นสี่ตัวหลักที่อยากพูดที่จริงพอเรามาดูทั้งหมดแล้วเนี่ยตั้งแต่เริ่มต้นที่เราพูดถึงเรื่องของการลดน้ําหนักไม่ว่าจะเป็นฟาสเต้งวิธีการกินต่าง ต่างเนาะรวมถึงเรื่อง ของฮอร์โมนด้วยเนี่ยคือจริงหมอก็คิดว่าคนไข้หรือใครที่สนใจอยากจะลดน้ําหนักที่จริงแนะนําอย่างแรกคือต้องตรวจก่อนต้องเจอแพทย์ก่อนต้องเจอแพทย์แล้วตรวจดูว่าคุณเนี่ยมีปัญหาเรื่องของฮอร์โมนหรือเปล่าของฮอร์โมนเพหญิงฮอร์โมนไทลอยหรือเปล่านะครับเนาะใช่หรือว่าบางทีอ่ะคนไข้อาจจะเป็นบอกว่าสารอาหารไม่พอหรือเปล่าใช่ให้เราคุยกับน้ําโปตีนไม่พออย่างเงี้ยคุณก็อ้วนง่ายคือการตรวจเนี่ยมันไม่ใช่ว่าเรามาแก้สาเหตุอย่างเดียวมันช่วยป้องกันด้วยไม่ให้ป้องกันที่ไม่ให้คุณเป็นโลกอ้วนน้ําหนักเกินหรือจนกระทั่งเป็นโรคเบาหวาน อืม เพราะฉะนั้นการตรวจกับการป้องกันเป็นอะไรที่สําคัญมาก จนสุดท้ายเนี่ยพออ่ะได้ผลมาแล้วเราอาจถึงหมออาจจะช่วยแนะนําวิธีที่ถูกต้องสําหรับตัวคุณและอาจจะพูดถึงเรื่องของยาหรืออาหารเสริมที่เหมาะกับตัวคุณ อือ เพราะว่าจริง จริงก็อยากจะฝากอีกอันนึงเหมือนกันว่าการที่เราทําอย่างเงี้ยมันเป็นอะไรที่สําคัญมากหนึ่งคือต้องหาสาเหตุให้เจอเนาะว่าคนไข้อ่ะอ้วนจากอะไรใช่ป่ะแก้ให้ถูกจึกแต่อันที่สองสําคัญมากคือคนไข้ต้องทําเองนะคือเนื่องจากว่าส่วนใหญ่คนไข้มาหมอคะหนูอยากขอ ค่ะแต่หนูอยากผอมโดยทางรัฐก็คือใช้ยา จะบอกว่าจริง จริงการใช้ยามันอาจจะไม่เป็นไรที่ยั่งยืนพฤติกรรมหรือไลฟ์สไตล์คนต่างหากที่สําคัญที่สุดเนาะ ก็บอกเลยว่าถ้าคนไข้เนี่ยจะมาบอกว่าเราก็เลยบอกว่าจริง จริงแล้วว่าหมอช่วยได้แค่ยี่สิบนะคุณต้องช่วยตัวเองแปดสิบเปอร์เซ็นต์ และนี่คือ ด็อกเตอร์สท podcast ที่หมอและผู้เชี่ยวชาญทางด้านสุขภาพจะมาพูดคุยประเด็นเรื่องสุขภาพต่าง ต่าง ถ้าชอบคอนเทนต์แนวนี้ฝากติดตามและ Subscribe เป็นกําลังใจให้หมอด้วยนะครับ สําหรับวันนี้ สวัสดีครับ สวัสดีค่ะ
